# Modelisation CTM

## Imports divers

In [ ]:
import pickle
import os
import numpy as np
import pandas as pd
from pathlib import Path

from contextualized_topic_models.utils.data_preparation import TopicModelDataPreparation
from contextualized_topic_models.models.ctm import CombinedTM
from sentence_transformers import SentenceTransformer

from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora import Dictionary

## Embeddings

In [ ]:
# Fonction pour charger ou générer des embeddings
# En cas de génération, les embeddings sont sauvegardés pour une utilisation future
def load_or_generate_embeddings(
    texts: list[str],
    model_name: str,
    emb_path: Path,
    force: bool = False,
    batch_size: int = 32
) -> np.ndarray:

    if emb_path.exists() and not force:
        print(f"🔹 Chargement des embeddings existants : {emb_path}")
        with open(emb_path, "rb") as f:
            X = pickle.load(f)

        X = np.asarray(X)

        if X.shape[0] != len(texts):
            raise ValueError(
                "Mismatch entre le nombre de textes et le nombre d'embeddings.\n"
                "Active FORCE_REGENERATE_EMBEDDINGS = True"
            )

        return X

    print("Génération des embeddings contextuels…")
    model = SentenceTransformer(model_name, trust_remote_code=True)

    X = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True
    )

    X = np.asarray(X)

    with open(emb_path, "wb") as f:
        pickle.dump(X, f)

    print(f"Embeddings sauvegardés dans {emb_path}")
    return X

Attention, la cellule ci-dessus peut prendre un certains temps pour s'executer si vous n'avez pas déjà le fichier d'embeddings, ou que vous forcez une nouvelle génération, **l'utilisation d'un GPU est fortement recommandé**.

In [ ]:
# Passer à true pour regénérer les embeddings (en cas de modifications du préprocessing par exemple)
FORCE_REGENERATE_EMBEDDINGS = False

df = pd.read_pickle(Path("../data/processed/dataset_final.pkl"))

texts_avec_contexte = df["clean_comment"].astype(str).tolist()
texts_sans_contexte = df["comment_sans_contexte"].astype(str).tolist()

X_ctx = load_or_generate_embeddings(
    texts=texts_avec_contexte,
    model_name="dangvantuan/french-document-embedding",
    emb_path=Path("../data/embeddings/emb_sbert_fr_ctm.pkl"),
    force=FORCE_REGENERATE_EMBEDDINGS,
    batch_size=32
)

X_ctx = np.asarray(X_ctx)
assert X_ctx.shape[0] == len(texts_avec_contexte), "Mismatch nb docs vs nb embeddings"

## Modelisation

In [ ]:
tp = TopicModelDataPreparation("dangvantuan/french-document-embedding")

training_dataset = tp.fit(
    text_for_contextual=texts_avec_contexte,
    text_for_bow=texts_sans_contexte,
    custom_embeddings=X_ctx
)

bow_size = len(tp.vocab)
ctx_size = X_ctx.shape[1]
bow_size, ctx_size

In [ ]:
K = 30 # nombre de topics
ctm = CombinedTM(
    bow_size=bow_size,
    contextual_size=ctx_size,
    n_components=K,
    num_epochs=20,
    num_data_loader_workers=0
)

ctm.fit(training_dataset) # entraînement

In [ ]:
doc_topic = ctm.get_doc_topic_distribution(training_dataset)  # shape (n_docs, K)
df["topic_id"] = np.argmax(doc_topic, axis=1)
df["topic_confidence"] = doc_topic.max(axis=1)

### Sauvegarde du modele

In [ ]:
os.makedirs(Path("../models/artifacts/ctm"), exist_ok=True)
os.makedirs(Path("../models/artifacts/ctm/exports"), exist_ok=True)

ctm.save(models_dir=Path("../models/artifacts/ctm/model"))

with open(Path("../models/artifacts/ctm/vocab.pkl"), "wb") as f:
    pickle.dump(tp.vocab, f)

np.save(Path("../models/artifacts/ctm/exports/doc_topic.npy"), doc_topic)

df.to_csv(Path("../models/artifacts/ctm/exports/reviews_with_topics.csv"), index=False)

## Affichage des résultats

In [ ]:
topics_words = ctm.get_topic_lists(20)  #  le nombre de mots à afficher par topic
for k, words in enumerate(topics_words):
    print(f"Topic {k}: {', '.join(words)}")

In [ ]:
# sauvegarde des top words par topic
pd.DataFrame({
    "topic_id": np.arange(len(topics_words)),
    "top_words": [", ".join(w) for w in topics_words]
}).to_csv(Path("../models/artifacts/ctm/exports/topics_top_words.csv"), index=False)

In [ ]:
topic_counts = df["topic_id"].value_counts().sort_index()
display(topic_counts)

In [ ]:
# Exemple de quelques commentaires par topic
def examples_for_topic(t, n=5):
    return df[df["topic_id"]==t]["clean_comment"].head(n).tolist()

for t in range(K):
    ex = examples_for_topic(t, 10)
    print(f"\n=== Topic {t} ===")
    for e in ex:
        print("-", e[:])


# 100 commentaires par catégories
# --- Définition des groupes de topics ---
# topics_livraison = [0, 3, 15, 17, 19, 20, 24, 27]
# topics_service_client = [1, 2, 6, 7, 8, 11, 14, 18, 21, 22, 23, 25, 28]
# topics_qualite_produit = [4, 9, 10, 12, 13, 16, 26, 29]

# # --- Fonction pour récupérer un nombre donné d'avis par catégorie ---
# def sample_by_topics(topic_list, n=100):
#     df_filtered = df[df["topic_id"].isin(topic_list)]
#     # si moins de 100 avis, on prend tout
#     return df_filtered["clean_comment"].sample(min(n, len(df_filtered)), random_state=42).tolist()

# # --- Récupération des 100 avis ---
# livraison_examples = sample_by_topics(topics_livraison, n=100)
# service_client_examples = sample_by_topics(topics_service_client, n=100)
# qualite_produit_examples = sample_by_topics(topics_qualite_produit, n=100)

# # --- Affichage propre ---
# def print_examples(label, examples):
#     print(f"\n=== {label} : {len(examples)} avis ===")
#     for e in examples:
#         print("-", e[:200])  # tronquer à 200 caractères pour lisibilité

# print_examples("SERVICE LIVRAISON", livraison_examples)
# print_examples("SERVICE CLIENT", service_client_examples)
# print_examples("QUALITÉ PRODUIT", qualite_produit_examples)


In [ ]:
#Évaluer la diversité des topics
def topic_diversity(topic_words):
    top_words = []
    for words in topic_words:
        top_words.extend(words)

    unique_words = set(top_words)
    diversity = len(unique_words) / len(top_words)
    return diversity
diversity_score = topic_diversity(topics_words)

print(f"Diversité des topics: {diversity_score:.2f}") # closer to 1 : each topic uses unique words(good diversity)

In [ ]:
# texts : liste de documents, chaque document = liste de tokens
# topic_words : liste de listes des mots top de chaque topic
texts = df["comment_sans_contexte"].apply(lambda x: x.split()).tolist()
dictionary = Dictionary(texts)

cm = CoherenceModel(
    topics=topics_words,
    texts=texts,
    dictionary=dictionary,
    coherence='c_v'
)

coherence_score = cm.get_coherence()

print("Score de cohérence:", coherence_score)

In [ ]:
for c in df[df["topic_id"] == 0]["Commentaire"]:
    print(c, "\n---\n")

In [ ]:
# Regroupement des topics
qualite_produit_topics = {3, 6, 18, 19, 23, 29}
livraison_topics = {5, 10, 24, 26, 27}
service_client_topics = {7, 8, 13, 14, 28}

df['label'] = np.nan

df.loc[df['topic_id'].isin(qualite_produit_topics), 'label'] = 0
df.loc[df['topic_id'].isin(livraison_topics), 'label'] = 1
df.loc[df['topic_id'].isin(service_client_topics), 'label'] = 2

df['label'] = df['label'].astype('Int64')  # int nullable
df['label'].value_counts(dropna=False)